**Contexte**

Une entreprise possède plusieurs bâtiments équipés de capteurs IoT.
Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la pression,
la consommation énergétique, le bâtiment, la date et l'heure de la mesure.
Chaque mesure possède également un état (OK, ALERTE et ERREUR).
L'objectif de l'atelier est de construire un modèle capable de prédire automatiquement l'état d'un
capteur à partir de ses mesures.
L'atelier suivra le workflow classique du Machine Learning :

`Dataset → Chargement → Exploration → Nettoyage → X / y → Train / Test → Prétraitement →
Modèle → fit()→ predict()→ Évaluation → Sauvegarde → Chargement → Réutilisation`

**Partie 0 – mise en place de l’environnement**

In [32]:
# importation des bibliothéques

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

Importation du dataset et vérification

In [26]:
df=pd.read_csv("../data/mesures_capteurs.csv")

df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    object 
 1   date_heure    605 non-null    object 
 2   id_capteur    605 non-null    object 
 3   batiment      605 non-null    object 
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    object 
dtypes: float64(4), object(5)
memory usage: 42.7+ KB


In [28]:
df.describe(include="all")

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
count,605,605,605,605,599.000000,600.00000,600.000000,600.000000,601
unique,600,600,12,4,NaN,NaN,NaN,NaN,3
top,M0302,2026-01-17 13:00:00,C002,B001,NaN,NaN,NaN,NaN,OK
freq,2,2,52,153,NaN,NaN,NaN,NaN,567
mean,NaN,NaN,NaN,NaN,24.878314,64.92620,1012.221900,208.675417,NaN
std,NaN,NaN,NaN,NaN,4.059576,10.76905,10.599042,72.243567,NaN
min,NaN,NaN,NaN,NaN,-18.500000,28.52000,850.000000,18.120000,NaN
25%,NaN,NaN,NaN,NaN,22.570000,58.17250,1006.790000,160.177500,NaN
50%,NaN,NaN,NaN,NaN,24.860000,65.37500,1012.855000,206.150000,NaN
75%,NaN,NaN,NaN,NaN,27.275000,71.61500,1017.827500,254.127500,NaN


**Partie 1 – Gestion des doublons**

1) vérifier l’existence de doublons dans df

In [33]:
# pour verifier l'existence de doublons on utilise la méthode duplicated la somme par sum 

nb_doublons = df.duplicated().sum()
print(f"Nombre de doublons détectés : {nb_doublons}")

Nombre de doublons détectés : 5


2) supprimer les doublons puis vérifier la suppression

In [35]:
print("Ancienne dimension du dataframe", df.shape)

# la m"thode drop_duplicates permet de supprimer les doublons dans un dataset
df = df.drop_duplicates()


print(f"Nombre de doublons après nettoyage : {df.duplicated().sum()}")
print("Nouvelle dimension du dataframe :", df.shape)

Ancienne dimension du dataframe (605, 9)
Nombre de doublons après nettoyage : 0
Nouvelle dimension du dataframe : (600, 9)


Comme on fait des étude je vais en profiter également pour retirer les lignes dont la **cible `etat` est manquante** :
il est en effet impossible d'entraîner ou d'évaluer un modèle supervisé sur une observation dont on
ne connaît pas la vraie classe.

In [36]:
print("Valeurs manquantes dans 'etat' avant nettoyage :", df['etat'].isna().sum())
df = df.dropna(subset=['etat']).reset_index(drop=True)

print("Valeurs manquantes dans 'etat' après nettoyage :", df['etat'].isna().sum())
print("Dimension finale du dataframe :", df.shape)

Valeurs manquantes dans 'etat' avant nettoyage : 4
Valeurs manquantes dans 'etat' après nettoyage : 0
Dimension finale du dataframe : (596, 9)


**Partie 2 – Sélection de y (cible) et X (caractéristiques)**

1) Définir "etat" comme la cible ou valeur à prédire et "temperature", "humidite",
"pression" et "consommation" comme caractéristiques ou variables explicatives

In [ ]:
# Alors ici les variables explicatives sont "temperature", "humidite",
#"pression" et "consommation"qui correspond aus features 
# la variable cible est etat donc "etat" est le target_name

features = ["temperature", "humidite", "pression", "consommation"]
target = "etat"

X = df[features]
y = df[target]


2) Afficher les cinq premières lignes de X et de y

In [45]:
#pour afficher on vas utiliser head

print("Affichage de des cinq premiéres lignes de X")
print(X.head())
print("Affichage de des cinq premiéres lignes de y")

print(y.head())

Affichage de des cinq premiéres lignes de X
   temperature  humidite  pression  consommation
0        25.46     58.06   1008.95        287.28
1        24.00     79.73    993.39        116.20
2        25.82     54.47   1010.32        288.50
3        28.23     69.39   1019.62        136.65
4        20.58     53.80   1016.58        182.62
Affichage de des cinq premiéres lignes de y
0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: object


3) Quel est le type du problème de machine learning ?

Il s'agit d'un problème d'**apprentissage supervisé de classification**, et plus précisément de
**classification multi-classes** (3 classes possibles : `OK`, `ALERTE`, `ERREUR`) car :

- on dispose d'exemples étiquetés (la colonne `etat`) → **apprentissage supervisé** ;
- la variable à prédire est **catégorielle** (et non continue) → **classification** (et non régression) ;
- il y a plus de deux modalités possibles `OK`, `ALERTE`, `ERREUR` → **classification multi-classes** (et non binaire).

**Partie 3 – Découpage Train/Test**

Diviser X en deux ensembles distincts : un pour l'entraînement (train) et un pour le test (test).
Avec les conditions suivantes : 20% des données serviront au test ; garantir la reproductibilité du
découpage ; conserver les mêmes proportions de classes dans l'ensemble de train et de test que
dans les données d'origine.

In [47]:
# d'abord nous allons importer sickit-learn aprés importer le modele et enfin importer la fonction train_test_split 

from sklearn.model_selection import train_test_split


X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("Taille X_train :", X_train.shape)
print("Taille X_test  :", X_test.shape)
print("\nProportions dans y_train :")
print(y_train.value_counts(normalize=True))
print("\nProportions dans y_test :")
print(y_test.value_counts(normalize=True))

Taille X_train : (476, 4)
Taille X_test  : (120, 4)

Proportions dans y_train :
etat
OK        0.947479
ALERTE    0.042017
ERREUR    0.010504
Name: proportion, dtype: float64

Proportions dans y_test :
etat
OK        0.925
ALERTE    0.075
Name: proportion, dtype: float64


**Partie 4 – Gestion des valeurs manquantes**

1) Vérifier l’existence de valeurs manquantes

In [48]:
# Pour verifier les valeurs manquantes on utilise la méthode isna dans le X_train

X_train.isna().sum()

temperature     6
humidite        5
pression        3
consommation    5
dtype: int64

2) Sélectionner SimpleImputer avec la médiane

In [49]:
# on utilise SimpleImputer et on met le affecte le  type distribution à la variable strategy comme parmaétre de SimpleImputer

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

3) Qu'est-ce qui justifie le choix de la médiane ?



- La médiane est **robuste aux valeurs extrêmes (outliers)** : contrairement à la moyenne, elle
  n'est pas influencée par quelques mesures aberrantes qu'un capteur défaillant pourrait produire.
- Les variables physiques mesurées (température, humidité, pression, consommation) peuvent avoir
  des distributions **non parfaitement symétriques** ; la médiane reste alors une valeur plus
  représentative de la tendance centrale que la moyenne.
- C'est une méthode simple, rapide et couramment utilisée comme choix par défaut robuste pour des
  données numériques.

4) Trouver les paramètres (médianes) de l’imputeur sur X_train

In [57]:
# Calcul des paramètres (médianes) de l'imputeur UNIQUEMENT sur X_train
# (on ne doit jamais apprendre les paramètres sur le test, pour éviter la fuite de données / data leakage)

print("Médianes apprises sur X_train :")
for col, med in zip(features, imputer.statistics_):
    print(f"  - {col} : {med}")


Médianes apprises sur X_train :
  - temperature : 24.845
  - humidite : 65.22
  - pression : 1013.05
  - consommation : 201.62


5) Déterminer X_train_imputed et X_test_imputed, les transformés de X_train et X_test

In [51]:
X_train_imputed = pd.DataFrame(imputer.transform(X_train), columns=features, index=X_train.index)
X_test_imputed  = pd.DataFrame(imputer.transform(X_test),  columns=features, index=X_test.index)

print("Valeurs manquantes restantes dans X_train_imputed :", X_train_imputed.isna().sum().sum())
print("Valeurs manquantes restantes dans X_test_imputed  :", X_test_imputed.isna().sum().sum())
X_train_imputed.head()

Valeurs manquantes restantes dans X_train_imputed : 0
Valeurs manquantes restantes dans X_test_imputed  : 0


,temperature,humidite,pression,consommation
426,25.910,49.20,1016.57,201.37
519,27.300,61.24,1011.93,168.13
137,24.845,66.77,1015.33,216.31
594,24.830,79.77,1017.05,170.97
319,29.440,55.30,1011.59,248.87


Partie 5 – Mise à l'échelle

1) Sélectionner StandardScaler pour mettre à l’échelle les transformés de l’imputation

In [52]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

2) Qu’est ce qui justifie la standardisation ?

- alors ici nos variables n'ont pas la même échelle : la `pression` est de l'ordre de 1000 (hPa) alors que
  l'`humidite` est de l'ordre de 0-100 (%). Sans standardisation, la pression écraserait totalement
  l'influence des autres variables dans le calcul de distance.
- La **standardisation** (moyenne = 0, écart-type = 1) met toutes les variables sur une échelle
  comparable, ce qui permet à chaque caractéristique de contribuer équitablement à la distance
  euclidienne utilisée par le KNN.

3) Trouver les paramètres (moyennes et écart-types) du scaleur sur X_train_imputed

In [64]:
print("Moyennes apprises :", dict(zip(features, scaler.mean_.round(2))))
print("Écarts-types appris :", dict(zip(features, scaler.scale_.round(2))))

Moyennes apprises : {'temperature': np.float64(24.84), 'humidite': np.float64(64.81), 'pression': np.float64(1012.38), 'consommation': np.float64(206.27)}
Écarts-types appris : {'temperature': np.float64(4.2), 'humidite': np.float64(10.68), 'pression': np.float64(11.18), 'consommation': np.float64(72.19)}


4) Déterminer X_train_scaled et X_test_scaled, les transformés de X_train_imputed et
X_test_imputed

In [65]:
X_train_scaled = pd.DataFrame(scaler.transform(X_train_imputed), columns=features, index=X_train_imputed.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test_imputed),  columns=features, index=X_test_imputed.index)

X_train_scaled.describe().round(2)

,temperature,humidite,pression,consommation
count,476.00,476.00,476.00,476.00
mean,0.00,-0.00,-0.00,0.00
std,1.00,1.00,1.00,1.00
min,-10.31,-2.75,-14.52,-2.61
25%,-0.54,-0.62,-0.48,-0.64
50%,0.00,0.04,0.06,-0.06
75%,0.58,0.61,0.51,0.60
max,8.05,7.51,2.33,9.26


Partie 6 – Entrainement et prédiction d’un modèle

1) Sélectionner le modèle KNN (k plus proches voisins) avec 5, le nombre de voisins à prendre en
compte

In [66]:
# alors le nombre de k voisins est déterminer à partir de n_neighbors

from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)

2) Entrainer le modèle

In [67]:
# Alors on Entraîne le modèle sur les données prétraitées avec fit
model.fit(X_train_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[object](3,)","['ALERTE','ERREUR','OK']"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


3) Déterminer y_pred, la prédiction du modèle avec l’ensemble de test

In [68]:
# on utilise predict pour faire une prediction et on le fait sur les features (X_test_scaled)
y_pred = model.predict(X_test_scaled)

4) Afficher quelques prédictions

In [69]:
# nous allons aficher les 15 premiére 
y_pred[:15]

array(['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK',
       'OK', 'OK', 'OK', 'OK'], dtype=object)

5) Comparer ces prédictions avec les vraies valeurs

In [ ]:
comparaison = pd.DataFrame({
    "valeur_reelle": y_test.values,
    "prediction":    y_pred
}).reset_index(drop=True)

# affichage des 15 premiéres
comparaison.head(15)

,valeur_reelle,prediction
0,OK,OK
1,OK,OK
2,OK,OK
3,OK,OK
4,OK,OK
5,OK,OK
6,OK,OK
7,ALERTE,OK
8,OK,OK
9,OK,OK


Partie 7 – Evaluation du modèle

1) Evaluer le modèle avec Accuracy puis afficher le résultat

In [ ]:
from sklearn.metrics import accuracy_score

# alors ici on utilise accuracy_score(la_vrai_valeur , valeur_predit)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy du modèle : {accuracy:.2%}")


Accuracy du modèle : 93.33%
